# GenAI – Prompt Engineering: LangChain Prompt Templates

**Objective:** Build dynamic and reusable prompt systems using LangChain instead of hardcoded prompts.

**Pipeline Flow:** User Input → Validation → Prompt Template → Dynamic Prompt Generation → Output

## Setup – Install and Import Required Libraries

In [5]:
!pip install langchain langchain-core

In [10]:
# Import required modules from LangChain
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate, SystemMessagePromptTemplate

---
## Task 1: Replace Hardcoded Prompts 

Convert the given f-string function into a reusable LangChain `PromptTemplate`.

In [12]:
template_1 = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

# Test
print(template_1.format(topic="AI"))

Explain AI in simple terms for beginners


---
## Task 2: Multi-Input Prompt System 

Build a template that accepts `topic`, `audience`, and `tone` as inputs.

In [15]:
template_2 = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

# Test cases
print(template_2.format(topic="AI", audience="beginners", tone="friendly"))
print(template_2.format(topic="Python", audience="kids", tone="fun"))
print(template_2.format(topic="Deep Learning", audience="engineers", tone="technical"))

Explain AI for beginners in a friendly tone
Explain Python for kids in a fun tone
Explain Deep Learning for engineers in a technical tone


---
## Task 3: Prompt Variations Engine 

Create 3 distinct `PromptTemplate` objects – Teaching, Interview, and Storytelling.

In [18]:
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 questions about {topic}"
)

story_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)

topic = "Machine Learning"

print(teaching_template.format(topic=topic))
print(interview_template.format(topic=topic))
print(story_template.format(topic=topic))

Explain Machine Learning clearly step by step
Ask 3 questions about Machine Learning
Explain Machine Learning as a story


---
## Task 4: ChatPromptTemplate System

Build a chat-based prompt with `System` (role behavior) and `User` (dynamic input) messages.

In [53]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}"),
    ("user", "Explain {topic}")
])

# Example
messages = chat_template.format_messages(role="teacher", topic="Neural Networks")

for msg in messages:
    print(msg)

content='You are a teacher' additional_kwargs={} response_metadata={}
content='Explain Neural Networks' additional_kwargs={} response_metadata={}


---
## Task 5: Input Validation Layer

Validate `audience` and `tone` inputs; assign defaults if values are invalid.

In [55]:
# ── Allowed values ──
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]

# Default fallback values
DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE     = "casual"

def validate_inputs(audience, tone, raise_on_invalid=False):
    """
    Validate audience and tone inputs.

    Parameters:
        audience (str): Target audience level.
        tone (str): Communication tone.
        raise_on_invalid (bool): If True, raises ValueError on invalid input.
                                 If False, assigns default values.

    Returns:
        (str, str): Validated (or defaulted) audience and tone.
    """
    validated_audience = audience.lower().strip()
    validated_tone     = tone.lower().strip()

    # Check audience
    if validated_audience not in VALID_AUDIENCES:
        if raise_on_invalid:
            raise ValueError(
                f"Invalid audience '{audience}'. Must be one of: {VALID_AUDIENCES}"
            )
        print(f"  ⚠ Invalid audience '{audience}' → defaulting to '{DEFAULT_AUDIENCE}'")
        validated_audience = DEFAULT_AUDIENCE

    # Check tone
    if validated_tone not in VALID_TONES:
        if raise_on_invalid:
            raise ValueError(
                f"Invalid tone '{tone}'. Must be one of: {VALID_TONES}"
            )
        print(f"  ⚠ Invalid tone '{tone}' → defaulting to '{DEFAULT_TONE}'")
        validated_tone = DEFAULT_TONE

    return validated_audience, validated_tone


# ── Test Validation ──
print("=== Input Validation Tests ===")

# Valid inputs
print("\nTest 1 – All valid:")
a, t = validate_inputs("expert", "formal")
print(f"  Result → audience: '{a}', tone: '{t}'")

# Invalid audience
print("\nTest 2 – Invalid audience:")
a, t = validate_inputs("student", "fun")
print(f"  Result → audience: '{a}', tone: '{t}'")

# Invalid tone
print("\nTest 3 – Invalid tone:")
a, t = validate_inputs("intermediate", "aggressive")
print(f"  Result → audience: '{a}', tone: '{t}'")

# Both invalid
print("\nTest 4 – Both invalid (defaults applied):")
a, t = validate_inputs("alien", "robotic")
print(f"  Result → audience: '{a}', tone: '{t}'")

# Raise error mode
print("\nTest 5 – Raise on invalid:")
try:
    validate_inputs("alien", "robotic", raise_on_invalid=True)
except ValueError as e:
    print(f"  Error caught: {e}")
print()

=== Input Validation Tests ===

Test 1 – All valid:
  Result → audience: 'expert', tone: 'formal'

Test 2 – Invalid audience:
  ⚠ Invalid audience 'student' → defaulting to 'beginner'
  Result → audience: 'beginner', tone: 'fun'

Test 3 – Invalid tone:
  ⚠ Invalid tone 'aggressive' → defaulting to 'casual'
  Result → audience: 'intermediate', tone: 'casual'

Test 4 – Both invalid (defaults applied):
  ⚠ Invalid audience 'alien' → defaulting to 'beginner'
  ⚠ Invalid tone 'robotic' → defaulting to 'casual'
  Result → audience: 'beginner', tone: 'casual'

Test 5 – Raise on invalid:
  Error caught: Invalid audience 'alien'. Must be one of: ['beginner', 'intermediate', 'expert']



---
## Task 6: Prompt Generator App 

Combine validation + templates into a single `generate_prompt()` function with style support.

In [57]:
# ── Style-aware PromptTemplates ──
style_templates = {
    "teaching":      PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} teaching style, step by step."
    ),
    "interview":     PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Generate interview questions about {topic} for {audience} with a {tone} tone."
    ),
    "storytelling":  PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} for {audience} in a {tone} storytelling style."
    ),
}

def generate_prompt(topic, audience, tone, style):
    """
    Full prompt generator with validation and style selection.

    Parameters:
        topic (str): The subject to explain.
        audience (str): Target audience (beginner / intermediate / expert).
        tone (str): Communication tone (formal / casual / fun).
        style (str): Prompt style (teaching / interview / storytelling).

    Returns:
        str: A fully formatted prompt string.
    """
    # Step 1: Validate audience and tone (defaults applied silently)
    validated_audience, validated_tone = validate_inputs(audience, tone)

    # Step 2: Validate style
    style_key = style.lower().strip()
    if style_key not in style_templates:
        raise ValueError(f"Unknown style '{style}'. Choose from: {list(style_templates.keys())}")

    # Step 3: Select the template and format it
    template = style_templates[style_key]
    return template.format(topic=topic, audience=validated_audience, tone=validated_tone)


# ── Test the Prompt Generator App ──
test_prompts = [
    {"topic": "Neural Networks",    "audience": "beginner",     "tone": "fun",      "style": "storytelling"},
    {"topic": "Python",             "audience": "intermediate", "tone": "casual",   "style": "teaching"},
    {"topic": "Data Structures",    "audience": "expert",       "tone": "formal",   "style": "interview"},
    {"topic": "Cloud Computing",    "audience": "student",      "tone": "friendly", "style": "teaching"},  # invalid audience & tone
]

print("=== Prompt Generator App ===")
for i, args in enumerate(test_prompts, 1):
    print(f"\nExample {i}:")
    result = generate_prompt(**args)
    print(f"  → {result}")
print()

=== Prompt Generator App ===

Example 1:
  → Explain Neural Networks for beginner in a fun storytelling style.

Example 2:
  → Explain Python for intermediate in a casual teaching style, step by step.

Example 3:
  → Generate interview questions about Data Structures for expert with a formal tone.

Example 4:
  ⚠ Invalid audience 'student' → defaulting to 'beginner'
  ⚠ Invalid tone 'friendly' → defaulting to 'casual'
  → Explain Cloud Computing for beginner in a casual teaching style, step by step.



---
## Task 7: Template Reusability Test 

Demonstrate one single `PromptTemplate` producing different outputs from 5 different inputs.

In [59]:
# ── ONE reusable template ──
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Provide a {tone} explanation of {topic} suitable for a {audience} audience."
)

# ── 5 different inputs for the same template ──
inputs = [
    {"topic": "Machine Learning",     "audience": "beginner",     "tone": "simple"},
    {"topic": "Blockchain",           "audience": "intermediate", "tone": "detailed"},
    {"topic": "Quantum Computing",    "audience": "expert",       "tone": "technical"},
    {"topic": "Natural Language Processing", "audience": "kids", "tone": "fun"},
    {"topic": "Computer Vision",      "audience": "professionals","tone": "concise"},
]

print("=== Template Reusability Test ===")
print(f"Template: '{reusable_template.template}'")
print("-" * 70)
for i, inp in enumerate(inputs, 1):
    output = reusable_template.format(**inp)
    print(f"Input {i}: topic='{inp['topic']}', audience='{inp['audience']}', tone='{inp['tone']}'")
    print(f"  Output → {output}")
    print()
print("✅ Same structure. Different outputs. Template is fully reusable.")

=== Template Reusability Test ===
Template: 'Provide a {tone} explanation of {topic} suitable for a {audience} audience.'
----------------------------------------------------------------------
Input 1: topic='Machine Learning', audience='beginner', tone='simple'
  Output → Provide a simple explanation of Machine Learning suitable for a beginner audience.

Input 2: topic='Blockchain', audience='intermediate', tone='detailed'
  Output → Provide a detailed explanation of Blockchain suitable for a intermediate audience.

Input 3: topic='Quantum Computing', audience='expert', tone='technical'
  Output → Provide a technical explanation of Quantum Computing suitable for a expert audience.

Input 4: topic='Natural Language Processing', audience='kids', tone='fun'
  Output → Provide a fun explanation of Natural Language Processing suitable for a kids audience.

Input 5: topic='Computer Vision', audience='professionals', tone='concise'
  Output → Provide a concise explanation of Computer Vision 

---
## Full Pipeline Demo

End-to-end demonstration: **User Input → Validation → Template → Output**

In [39]:
def run_pipeline(topic, audience, tone, style):
    """
    Full pipeline: Input → Validation → PromptTemplate → Output.
    """
    print("INPUT  → topic='{}', audience='{}', tone='{}', style='{}'".format(
        topic, audience, tone, style
    ))
    
    prompt = generate_prompt(topic, audience, tone, style)
    
    print("OUTPUT → {}".format(prompt))
    print()
    
print("=" * 70)
print("            COMPLETE PIPELINE DEMONSTRATION")
print("=" * 70)

run_pipeline("Reinforcement Learning", "expert",      "formal",  "interview")
run_pipeline("Python",                 "beginner",    "fun",     "teaching")
run_pipeline("Neural Networks",        "intermediate","casual",  "storytelling")

            COMPLETE PIPELINE DEMONSTRATION
INPUT  → topic='Reinforcement Learning', audience='expert', tone='formal', style='interview'
OUTPUT → Explain Reinforcement Learning for expert in a formal interview style

INPUT  → topic='Python', audience='beginner', tone='fun', style='teaching'
OUTPUT → Explain Python for beginner in a fun teaching style

INPUT  → topic='Neural Networks', audience='intermediate', tone='casual', style='storytelling'
OUTPUT → Explain Neural Networks for intermediate in a casual storytelling style



---
## Summary

| Task | Description | Status |
|------|-------------|--------|
| 1    | Replace Hardcoded Prompts with PromptTemplate | ✅ |
| 2    | Multi-Input Prompt System (topic, audience, tone) | ✅ |
| 3    | Prompt Variations Engine (Teaching, Interview, Storytelling) | ✅ |
| 4    | ChatPromptTemplate System with roles | ✅ |
| 5    | Input Validation Layer (with defaults & error raising) | ✅ |
| 6    | Prompt Generator App with style selection | ✅ |
| 7    | Template Reusability Test (1 template, 5 inputs) | ✅ |

**Key Principles Applied:**
- Used `input_variables` throughout — no f-strings
- No hardcoded values — all logic is modular and separated from templates
- Templates are reusable across multiple topics, audiences, and tones
- Validation layer protects prompt integrity before template formatting